In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
df=files.upload()

Saving 02_nav_history_cleaned.csv to 02_nav_history_cleaned.csv
Saving 08_investor_transactions_cleaned.csv to 08_investor_transactions_cleaned.csv
Saving 07_scheme_performance_cleaned.csv to 07_scheme_performance_cleaned.csv
Saving 01_fund_master - Copy.csv to 01_fund_master - Copy.csv
Saving 03_aum_by_fund_house.csv to 03_aum_by_fund_house.csv
Saving 04_monthly_sip_inflows.csv to 04_monthly_sip_inflows.csv
Saving 05_category_inflows.csv to 05_category_inflows.csv
Saving 06_industry_folio_count.csv to 06_industry_folio_count.csv
Saving 09_portfolio_holdings.csv to 09_portfolio_holdings.csv
Saving 10_benchmark_indices.csv to 10_benchmark_indices.csv


In [4]:
fund_master = pd.read_csv("01_fund_master - Copy.csv")
nav_history = pd.read_csv("02_nav_history_cleaned.csv")
aum_by_fund_house = pd.read_csv("03_aum_by_fund_house.csv")
monthly_sip_inflows = pd.read_csv("04_monthly_sip_inflows.csv")
category_inflows = pd.read_csv("05_category_inflows.csv")
industry_folio_count = pd.read_csv("06_industry_folio_count.csv")
scheme_performance = pd.read_csv("07_scheme_performance_cleaned.csv")
investor_transactions = pd.read_csv("08_investor_transactions_cleaned.csv")
portfolio_holdings = pd.read_csv("09_portfolio_holdings.csv")
benchmark_indices = pd.read_csv("10_benchmark_indices.csv")

**Compute Daily Returns**

In [8]:
nav_history =  nav_history.sort_values(["amfi_code","date"])
nav_history["daily_returns"] = nav_history.groupby("amfi_code")["nav"].pct_change()

In [12]:
print(nav_history["daily_returns"])

0             NaN
1       -0.010306
2        0.012865
3       -0.011377
4       -0.001210
           ...   
45995    0.012106
45996   -0.004138
45997   -0.008480
45998   -0.028093
45999   -0.003335
Name: daily_returns, Length: 46000, dtype: float64


**CAGR**

In [22]:
nav_history["date"]=pd.to_datetime(nav_history["date"])
nav_history=nav_history.sort_values(["amfi_code","date"])
cagr_list=[]
for amfi_code,group in nav_history.groupby("amfi_code"):
    start_nav=group.iloc[0]["nav"]
    end_nav=group.iloc[-1]["nav"]
    start_date=group.iloc[0]["date"]
    end_date=group.iloc[-1]["date"]
    years=(end_date-start_date).days/365.25
    cagr=((end_nav/start_nav)**(1/years)-1)*100 if years>0 else np.nan
    cagr_list.append([amfi_code,start_nav,end_nav,round(years,2),round(cagr,2)])
cagr_df=pd.DataFrame(cagr_list,columns=["amfi_code","start_nav","end_nav","years","cagr_pct"])
cagr_df.to_csv("fund_cagr.csv",index=False)

**SHARPE RATIO**

In [23]:
rf = 0.065

sharpe = ((nav_history["daily_returns"].mean() - rf/252)/nav_history["daily_returns"].std()) * np.sqrt(252)

In [24]:
sharpe

np.float64(0.5755862237652258)

**SORTINO RATIO**

In [25]:
nav_history["daily_return"]=nav_history.groupby("amfi_code")["nav"].pct_change()
sortino_list=[]
for amfi_code,group in nav_history.groupby("amfi_code"):
    returns=group["daily_return"].dropna()
    downside=returns[returns<0]
    if len(downside)>0:
        sortino=(returns.mean()/(downside.std()))*np.sqrt(252)
    else:
        sortino=np.nan
    sortino_list.append([amfi_code,round(sortino,2)])

sortino_df=pd.DataFrame(sortino_list,columns=["amfi_code","sortino_ratio"])
sortino_df.to_csv("sortino_ratio.csv",index=False)

**MAXIMUM DRAWDOWN**

In [26]:
running_max = nav_history['nav'].cummax()
drawdown = nav_history['nav']/running_max - 1
max_drawdown = drawdown.min()

**alpha and beta**

In [32]:
from scipy.stats import linregress
nav_history["daily_return"]=nav_history.groupby("amfi_code")["nav"].pct_change()

benchmark_indices["date"]=pd.to_datetime(benchmark_indices["date"])
benchmark_indices=benchmark_indices.sort_values("date")
benchmark_indices["benchmark_return"]=benchmark_indices["close_value"].pct_change()

alpha_beta=[]

for amfi_code,group in nav_history.groupby("amfi_code"):
    df=group.merge(benchmark_indices[["date","benchmark_return"]],on="date",how="inner").dropna()
    if len(df)>2:
        beta,alpha,_,_,_=linregress(df["benchmark_return"],df["daily_return"])
        alpha=alpha*252
    else:
        alpha=np.nan
        beta=np.nan
    alpha_beta.append([amfi_code,round(alpha,4),round(beta,4)])

alpha_beta_df=pd.DataFrame(alpha_beta,columns=["amfi_code","alpha","beta"])

**FUND-SCORE**

In [35]:
fund_scorecard=scheme_performance.merge(alpha_beta_df,on="amfi_code",how="left")
fund_scorecard["return_rank"]=fund_scorecard["return_3yr_pct"].rank(ascending=False)
fund_scorecard["sharpe_rank"]=fund_scorecard["sharpe_ratio"].rank(ascending=False)
fund_scorecard["alpha_rank"]=fund_scorecard["alpha_y"].rank(ascending=False)
fund_scorecard["expense_rank"]=fund_scorecard["expense_ratio_pct"].rank(ascending=True)
fund_scorecard["drawdown_rank"]=fund_scorecard["max_drawdown_pct"].rank(ascending=True)
fund_scorecard["fund_score"]=(
fund_scorecard["return_rank"]*0.30+
fund_scorecard["sharpe_rank"]*0.25+
fund_scorecard["alpha_rank"]*0.20+
fund_scorecard["expense_rank"]*0.15+
fund_scorecard["drawdown_rank"]*0.10
)

In [36]:
fund_scorecard=fund_scorecard.sort_values("fund_score")
fund_scorecard.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha_x,...,morningstar_rating,risk_grade,alpha_y,beta_y,return_rank,sharpe_rank,alpha_rank,expense_rank,drawdown_rank,fund_score
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,...,5,Very High,0.3026,0.0,1.0,18.0,1.0,21.0,33.0,11.450
12,120505,ICICI Pru Midcap Fund - Regular - Growth,ICICI Prudential MF,Mid Cap,Regular,14.02,18.08,17.55,17.19,0.89,...,3,High,0.2906,0.0,8.0,17.0,4.0,15.0,18.0,11.500
22,120843,Kotak Flexicap Fund - Regular - Growth,Kotak Mahindra MF,Flexi Cap,Regular,15.74,15.65,13.50,13.80,1.85,...,5,Moderately High,0.2578,0.0,11.0,12.0,6.0,22.0,21.0,12.900
11,120504,ICICI Pru Bluechip Fund - Direct - Growth,ICICI Prudential MF,Large Cap,Direct,14.12,14.41,13.02,13.53,0.88,...,3,Moderate,0.2217,-0.0,20.0,9.0,10.0,12.0,9.0,12.950
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset MF,Large Cap,Regular,15.12,14.81,12.68,13.19,1.62,...,5,Moderate,0.2665,0.0,17.0,7.5,5.0,23.0,26.0,14.025


In [37]:
alpha_beta_df.to_csv("alpha_beta.csv", index=False)
from google.colab import files
files.download("alpha_beta.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
fund_scorecard.to_csv("fund_scorecard.csv", index=False)
from google.colab import files
files.download("fund_scorecard.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>